# MoVE-KD trên Colab A100: eval, trực quan hoá, smoke-test train

Notebook cho phần việc của **Thành viên 3** (xem `Experiments.md`):

1. Cài môi trường (Python 3.10, torch 2.1.2, transformers 4.37.2, DeepSpeed, Flash-Attention) bằng `setup.sh`
2. Tải checkpoint `jiaojuncao/MoVE-KD-7b-v1.1` và dữ liệu POPE, MME, TextVQA, ScienceQA
3. Kiểm tra bộ chấm điểm bằng câu trả lời có sẵn của LLaVA-1.5-13B (không cần GPU)
4. Eval MoVE-KD-7B-v1.1 trên 4 benchmark, rồi tổng hợp bảng và biểu đồ
5. Vẽ [CLS] attention và routing của MoLE (Fig 5)
6. *(Tuỳ chọn)* Baseline LLaVA-1.5-7B bằng code LLaVA gốc
7. *(Tuỳ chọn)* Smoke-test Stage 1 với TinyLlama và 4 teacher

**Trước khi chạy:** chọn *Runtime → Change runtime type → A100 GPU*. Code được clone từ GitHub, nên các script trong `scripts/move-kd/` phải được push lên repo trước.

Thư mục làm việc `/content` sẽ bị xoá khi phiên kết thúc. Kết quả (answers, score, hình) được sao lưu vào Google Drive sau mỗi bước bằng `save()`.

## 0. Cấu hình

In [ ]:
REPO_URL   = "https://github.com/omninguyen/MoVE-KD"
BRANCH     = "tv3-colab-eval"   # đổi thành "main" sau khi merge
WORK       = "/content/MoVE-KD"
VENV       = "/content/venv"
DRIVE_OUT  = "/content/drive/MyDrive/MoVE-KD-TV3"   # nơi sao lưu kết quả
CKPT       = "./checkpoints/llava-move-kd-7b-v1.1"  # tên phải chứa 'llava'
NAME       = "llava-move-kd-7b-v1.1"

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os, shutil, glob, time
os.makedirs(DRIVE_OUT, exist_ok=True)

def save():
    """Sao lưu answers / score / hình sang Drive (gọi sau mỗi bước)."""
    ev = f"{WORK}/playground/data/eval"
    patterns = ["pope/answers/*", "textvqa/answers/*", "scienceqa/answers/*", "MME/answers/*",
                "MME/eval_tool/answers/*"]
    n = 0
    for p in patterns:
        for f in glob.glob(f"{ev}/{p}"):
            dst = os.path.join(DRIVE_OUT, "eval", os.path.relpath(f, ev))
            os.makedirs(os.path.dirname(dst), exist_ok=True)
            (shutil.copytree(f, dst, dirs_exist_ok=True) if os.path.isdir(f) else shutil.copy2(f, dst)); n += 1
    for f in glob.glob(f"{WORK}/images/*.png") + glob.glob(f"{WORK}/checkpoints/smoke/*.log"):
        os.makedirs(f"{DRIVE_OUT}/images", exist_ok=True); shutil.copy2(f, f"{DRIVE_OUT}/images/"); n += 1
    print(f"[ OK ] Đã sao lưu {n} mục -> {DRIVE_OUT}")

## 1. Môi trường

Colab dùng Python 3.12, nhưng `torch==2.1.2` chỉ có bản cài sẵn tới Python 3.11. Vì vậy notebook tạo một venv Python 3.10 riêng bằng `uv` (`--seed` để venv có sẵn pip), rồi chạy `setup.sh` với `SKIP_CONDA=1`. Mọi lệnh `!` và `%%bash` phía sau đều dùng venv này nhờ `PATH`. Bước này mất khoảng 10–15 phút.

In [ ]:
!git clone -b {BRANCH} {REPO_URL} {WORK}
%cd {WORK}
!git log --oneline -1

In [ ]:
!pip install -q uv
!uv venv --seed -p 3.10 {VENV}
os.environ["VIRTUAL_ENV"] = VENV
os.environ["PATH"] = f"{VENV}/bin:" + os.environ["PATH"]
!which python && python --version

In [ ]:
!SKIP_CONDA=1 bash setup.sh
!python -m pip install -q gdown remotezip

## 2. Checkpoint & dữ liệu eval

In [ ]:
# ~14GB, lưu vào checkpoints/llava-move-kd-7b-v1.1 (tên phải chứa 'llava', xem download_ckpt.sh)
!bash scripts/move-kd/eval/download_ckpt.sh

Dữ liệu theo [Evaluation.md của LLaVA](https://github.com/haotian-liu/LLaVA/blob/main/docs/Evaluation.md):

| Benchmark | Nguồn | Dung lượng |
| :--- | :--- | ---: |
| Tất cả | `eval.zip` (Google Drive của LLaVA): file câu hỏi, script convert MME, câu trả lời của LLaVA-1.5-13B | 23MB |
| POPE | Nhãn `coco_pope_*.json` (repo POPE) + ảnh COCO `val2014.zip` | ~6.2GB |
| TextVQA | `TextVQA_0.5.1_val.json` + `train_val_images.zip` | ~7GB |
| ScienceQA | `problems.json`, `pid_splits.json` + ảnh `test.zip` (S3 của ScienceQA) | ~0.4GB |
| MME | `MME_Benchmark_release_version.zip` (HF `darkyarding/MME`) + `eval_tool.zip` (repo Awesome-MLLM) | ~0.2GB |

Zip của MME trên HF lồng thêm một cấp `MME_Benchmark/`, nên cell dưới dời các thư mục category lên một cấp để khớp với cấu trúc LLaVA cần.

In [ ]:
%%bash
set -e
E=playground/data/eval
mkdir -p $E && cd $E

# eval.zip: câu hỏi đã định dạng cho LLaVA + câu trả lời của LLaVA-1.5-13B
[ -f pope/llava_pope_test.jsonl ] || { gdown -q 1atZSBBrAX54yYpxtVVW33zFvcnaHeFPy -O eval.zip && unzip -q -o eval.zip && rm eval.zip; }

# POPE
mkdir -p pope/coco
for s in random popular adversarial; do
  wget -q -nc -O pope/coco/coco_pope_$s.json \
    https://raw.githubusercontent.com/AoiDragon/POPE/e3e39262c85a6a83f26cf5094022a782cb0df58d/output/coco/coco_pope_$s.json
done
[ -d pope/val2014 ] || { wget -q http://images.cocodataset.org/zips/val2014.zip -O /content/val2014.zip && unzip -q /content/val2014.zip -d pope && rm /content/val2014.zip; }

# TextVQA
wget -q -nc -O textvqa/TextVQA_0.5.1_val.json https://dl.fbaipublicfiles.com/textvqa/data/TextVQA_0.5.1_val.json
[ -d textvqa/train_images ] || { wget -q https://dl.fbaipublicfiles.com/textvqa/images/train_val_images.zip -O /content/tv.zip && unzip -q /content/tv.zip -d textvqa && rm /content/tv.zip; }

# ScienceQA
wget -q -nc -O scienceqa/problems.json   https://raw.githubusercontent.com/lupantech/ScienceQA/main/data/scienceqa/problems.json
wget -q -nc -O scienceqa/pid_splits.json https://raw.githubusercontent.com/lupantech/ScienceQA/main/data/scienceqa/pid_splits.json
[ -d scienceqa/images/test ] || { mkdir -p scienceqa/images && wget -q https://scienceqa.s3.us-west-1.amazonaws.com/images/test.zip -O /content/sqa.zip && unzip -q /content/sqa.zip -d scienceqa/images && rm /content/sqa.zip; }

# MME
cd MME
if [ ! -d MME_Benchmark_release_version/artwork ]; then
  wget -q https://huggingface.co/datasets/darkyarding/MME/resolve/main/MME_Benchmark_release_version.zip -O mme.zip
  unzip -q -o mme.zip && rm mme.zip
  if [ -d MME_Benchmark_release_version/MME_Benchmark ]; then
    mv MME_Benchmark_release_version/MME_Benchmark/* MME_Benchmark_release_version/ && rmdir MME_Benchmark_release_version/MME_Benchmark
  fi
fi
[ -d eval_tool ] || { wget -q https://github.com/BradyFU/Awesome-Multimodal-Large-Language-Models/raw/Evaluation/tools/eval_tool.zip -O et.zip && unzip -q et.zip && rm et.zip; }
cd ..

echo "POPE    : $(ls pope/val2014 | wc -l) ảnh, $(wc -l < pope/llava_pope_test.jsonl) câu"
echo "TextVQA : $(ls textvqa/train_images | wc -l) ảnh, $(wc -l < textvqa/llava_textvqa_val_v051_ocr.jsonl) câu"
echo "SQA     : $(ls scienceqa/images/test | wc -l) thư mục ảnh test"
echo "MME     : $(ls MME/MME_Benchmark_release_version | wc -l) category, $(wc -l < MME/llava_mme.jsonl) câu"
df -h /content | tail -1

## 3. Kiểm tra bộ chấm điểm (không cần GPU)

Chấm lại các câu trả lời có sẵn của **LLaVA-1.5-13B** trong `eval.zip`. Kết quả phải khớp với Table 1 của paper: POPE ≈ 85.9, MME ≈ 1531.3, TextVQA ≈ 61.3, SQA-IMG ≈ 71.6. Nếu khớp, dữ liệu đáp án và bộ chấm đều đúng, và mọi chênh lệch ở bước 4 sẽ chỉ đến từ việc chạy model. Lần test trên máy local đã cho ra 85.95 / 1531.3 / 61.25 / 71.64.

In [ ]:
%%bash
set -e
N=llava-v1.5-13b
E=./playground/data/eval
python llava/eval/eval_pope.py --annotation-dir $E/pope/coco --question-file $E/pope/llava_pope_test.jsonl \
    --result-file $E/pope/answers/$N.jsonl > $E/pope/answers/${N}_score.txt
python -m llava.eval.eval_textvqa --annotation-file $E/textvqa/TextVQA_0.5.1_val.json \
    --result-file $E/textvqa/answers/$N.jsonl > $E/textvqa/answers/${N}_score.txt 2>/dev/null
python llava/eval/eval_science_qa.py --base-dir $E/scienceqa --result-file $E/scienceqa/answers/$N.jsonl \
    --output-file $E/scienceqa/answers/${N}_output.jsonl --output-result $E/scienceqa/answers/${N}_result.json \
    > $E/scienceqa/answers/${N}_score.txt
(cd $E/MME && python convert_answer_to_mme.py --experiment $N && cd eval_tool && \
    python calculation.py --results_dir answers/$N > ../answers/${N}_score.txt)
python - <<'EOF'
import importlib.util
spec = importlib.util.spec_from_file_location("s", "scripts/move-kd/summarize_results.py")
s = importlib.util.module_from_spec(spec); spec.loader.exec_module(s)
got = s.read_scores("llava-v1.5-13b")
ref = dict(zip(s.BENCHMARKS, s.TABLE1["13B"]["LLaVA-1.5"]))
for k, v in got.items():
    print(f"{k:8s} tự chấm {v:8.2f} | paper {ref[k]:7.1f}")
EOF

## 4. Eval MoVE-KD-7B-v1.1

Chạy thử 50 câu POPE trước để chắc model nạp đúng. Log phải có `EvalMoVELlavaLlamaForCausalLM` và không có lỗi `image_processor`. Sau đó chạy đủ 4 benchmark; mình ước lượng (chưa đo) khoảng 1 giờ trên A100.

In [ ]:
!python -c "from llava.model.builder import load_pretrained_model as L; t,m,ip,_=L('{CKPT}', None, '{NAME}'); print(type(m).__name__, type(ip).__name__)"

In [ ]:
!LIMIT=50 bash scripts/move-kd/eval/pope.sh {CKPT}

In [ ]:
timing = {}
for bench in ["pope", "mme", "textvqa", "sqa"]:
    t0 = time.time()
    get_ipython().system(f"bash scripts/move-kd/eval/{bench}.sh {CKPT} 2>&1 | tail -25")
    timing[bench] = (time.time() - t0) / 60
    print(f"===== {bench}: {timing[bench]:.1f} phút =====")
    save()
print(timing)

In [ ]:
!python scripts/move-kd/summarize_results.py --name {NAME} --paper-ref MoVE-KD-v1.1
save()

In [ ]:
from IPython.display import Image, display
for f in ["results_reproduced_" + NAME, "results_table1"]:
    display(Image(f"images/{f}.png"))

## 5. Trực quan hoá [CLS] attention và routing MoLE (Fig 5)

Dùng 2 ảnh mẫu của LLaVA, 2 ảnh COCO (POPE) và 2 ảnh TextVQA. Có thể thay bằng ảnh tuỳ ý qua tham số `--images`.

In [ ]:
imgs = (sorted(glob.glob("llava/serve/examples/*.jpg"))
        + sorted(glob.glob("playground/data/eval/pope/val2014/*.jpg"))[:2]
        + sorted(glob.glob("playground/data/eval/textvqa/train_images/*.jpg"))[:2])
img_args = " ".join(imgs)
!python scripts/move-kd/visualize_attention.py --ckpt {CKPT} --images {img_args}
save()
for f in sorted(glob.glob("images/attn_*.png")) + sorted(glob.glob("images/routing_*.png")):
    display(Image(f))

## 6. *(Tuỳ chọn)* Baseline LLaVA-1.5-7B

Code của repo này **không** eval được checkpoint LLaVA-1.5 gốc. `builder.py` nạp mọi model có tên `llava` bằng `EvalMoVELlavaLlamaForCausalLM`, nên CLIP bị gắn các expert LoRA ngẫu nhiên (xem `Experiments.md`, mục 2.6). Vì vậy cell dưới đặt code LLaVA gốc lên trước trong `PYTHONPATH`. Các wrapper eval vẫn dùng được nguyên vì cùng gọi `python -m llava.eval...`. Mất thêm khoảng 1 giờ.

In [ ]:
RUN_BASELINE = False
if RUN_BASELINE:
    !git clone -q https://github.com/haotian-liu/LLaVA /content/LLaVA
    for bench in ["pope", "mme", "textvqa", "sqa"]:
        get_ipython().system(f"PYTHONPATH=/content/LLaVA bash scripts/move-kd/eval/{bench}.sh liuhaotian/llava-v1.5-7b 2>&1 | tail -15")
        save()
    !python scripts/move-kd/summarize_results.py --name llava-v1.5-7b --paper-ref LLaVA-1.5

## 7. *(Tuỳ chọn)* Smoke-test Stage 1: 4 teacher + MoLE + adapters, LLM TinyLlama

Mục tiêu là chứng minh pipeline huấn luyện chạy được trên 1 GPU, không phải để tái lập số liệu.
- Tải checkpoint teacher SAM-L và EVA-02-L về đúng đường dẫn hardcode trong `mixture_vision_models.py`. ConvNeXt do timm tự tải khi chạy.
- `images.zip` của LCS-558K nặng 27GB, nên cell dùng `remotezip` để chỉ tải đúng `N_SAMPLES` ảnh cần dùng (đọc từng phần của zip qua HTTP range).

In [ ]:
RUN_SMOKE = False
N_SAMPLES = 200
if RUN_SMOKE:
    %pip install -q remotezip   # cài cho kernel Colab (cell này chạy trong kernel, không phải venv)
    from huggingface_hub import hf_hub_download, hf_hub_url, snapshot_download
    snapshot_download("facebook/sam-vit-large", local_dir="checkpoints/sam/sam_vit_l",
                      ignore_patterns=["*.h5", "*.msgpack", "*.bin"])
    hf_hub_download("Yuxin-CV/EVA-02", "eva02/det/eva02_L_coco_det_sys_o365.pth", local_dir="checkpoints/EVA")

    import json
    from remotezip import RemoteZip
    pre = "playground/data/LLaVA-Pretrain"
    hf_hub_download("liuhaotian/LLaVA-Pretrain", "blip_laion_cc_sbu_558k.json", repo_type="dataset", local_dir=pre)
    need = [d["image"] for d in json.load(open(f"{pre}/blip_laion_cc_sbu_558k.json"))[:N_SAMPLES]]
    with RemoteZip(hf_hub_url("liuhaotian/LLaVA-Pretrain", "images.zip", repo_type="dataset")) as z:
        for p in need:
            z.extract(p, f"{pre}/images")
    print(f"[ OK ] {len(need)} ảnh -> {pre}/images")

In [ ]:
if RUN_SMOKE:
    !N_SAMPLES={N_SAMPLES} MAX_STEPS=30 bash scripts/move-kd/smoke_pretrain.sh 2>&1 | tail -40
    !nvidia-smi --query-gpu=memory.used --format=csv
    save()

## 8. Kết quả

Toàn bộ answers, score và hình nằm ở thư mục `DRIVE_OUT` trên Drive. Điền vào `Experiments.md`:
- **Mục 2.5:** bảng do `summarize_results.py` in ra, thời gian chạy (`timing`), loại GPU
- **Mục 2.4:** các hình `attn_*.png`, `routing_*.png`, `routing_share.png`, kèm nhận xét

In [ ]:
print(timing if "timing" in globals() else "chưa chạy bước 4")
!ls -la {DRIVE_OUT} {DRIVE_OUT}/images